<a href="https://colab.research.google.com/github/saikarthikchegu/Speech-Emotion-Detection-and-Sentiment-Analysis./blob/main/NLP_Course_End_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Emotion Detection and Sentiment Analysis in Speech.
pip install numpy pandas librosa scipy nltk scikit-learn tensorflow

In [5]:
import os
import re
import numpy as np
import pandas as pd
import librosa
import scipy.signal as signal
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Flatten, Dense,
                                     LSTM, Dropout, concatenate)
from tensorflow.keras.utils import to_categorical
nltk.download('stopwords')
nltk.download('wordnet')
#Set random seed
np.random.seed(42)
tf.random.set_seed(42)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [6]:
#Section 1 Audio Processing
def reduce_noise_bandpass(audio, sr, lowcut=300.0, highcut=3400.0, order=5):
    """Applies a Butterworth bandpass filter to remove background noise."""
    nyquist = 0.5 * sr
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = signal.butter(order, [low, high], btype='band')
    filtered_audio = signal.filtfilt(b, a, audio)
    return filtered_audio

def extract_mfcc(file_path, max_pad_len=174, n_mfcc=40):
    """Loads audio, applies noise reduction, and extracts MFCC features."""
    try:
        # Load audio file (standardized sample rate)
        audio, sr = librosa.load(file_path, sr=22050)

        # 1. Noise Reduction
        clean_audio = reduce_noise_bandpass(audio, sr)

        # 2. MFCC Feature Extraction
        mfccs = librosa.feature.mfcc(y=clean_audio, sr=sr, n_mfcc=n_mfcc)

        # Pad or truncate MFCC sequence for uniform neural net input shape
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [8]:
#Section 2 Emotion Detection
def build_cnn_emotion_model(input_shape, num_classes):
    """Builds 1D Convolutional Neural Network for Audio Emotion Detection."""
    inputs = Input(shape=input_shape)
    x = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)

    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs, name="CNN_Emotion_Model")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_rnn_emotion_model(input_shape, num_classes):
    """Builds Recurrent Neural Network (LSTM) for Audio Emotion Detection."""
    inputs = Input(shape=input_shape)
    x = LSTM(128, return_sequences=True)(inputs)
    x = Dropout(0.3)(x)
    x = LSTM(64)(x)
    x = Dropout(0.3)(x)

    x = Dense(64, activation='relu')(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs, name="RNN_LSTM_Emotion_Model")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [9]:
#Section 3 Sentiment Analysis..
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Applies cleaning, tokenization, stopword removal, and lemmatization."""
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    # Tokenization & Stopword removal + Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

def train_text_classifiers(X_train_vec, y_train, X_test_vec, y_test):
    """Trains and evaluates traditional supervised ML algorithms on text."""
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Support Vector Machine": SVC(kernel='linear'),
        "Random Forest": RandomForestClassifier(n_estimators=100)
    }

    results = {}
    for name, clf in models.items():
        clf.fit(X_train_vec, y_train)
        preds = clf.predict(X_test_vec)
        acc = accuracy_score(y_test, preds)
        p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')

        results[name] = {
            "model": clf,
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1_score": f1
        }
        print(f"[{name}] Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")

    return results

In [10]:
#Section 4 Multi Modal Integration..
def build_multimodal_model(audio_shape, text_dim, num_classes):
    """Integrates audio MFCCs (CNN stream) and text features (Dense stream)."""
    # Stream 1: Audio Feature Extraction (CNN)
    audio_input = Input(shape=audio_shape, name="Audio_Input")
    x_audio = Conv1D(64, kernel_size=3, activation='relu')(audio_input)
    x_audio = MaxPooling1D(pool_size=2)(x_audio)
    x_audio = Flatten()(x_audio)
    x_audio = Dense(64, activation='relu')(x_audio)

    # Stream 2: Text Feature Extraction (Dense representation of TF-IDF)
    text_input = Input(shape=(text_dim,), name="Text_Input")
    x_text = Dense(128, activation='relu')(text_input)
    x_text = Dropout(0.3)(x_text)
    x_text = Dense(64, activation='relu')(x_text)

    # Early/Mid Fusion: Concatenate both feature streams
    fused = concatenate([x_audio, x_text])
    fused_dense = Dense(64, activation='relu')(fused)
    fused_dense = Dropout(0.3)(fused_dense)

    # Output Layer
    output = Dense(num_classes, activation='softmax', name="Multimodal_Output")(fused_dense)

    model = Model(inputs=[audio_input, text_input], outputs=output, name="Multimodal_Fusion_Model")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [11]:
#Final Demo Execution Pipeline..
if __name__ == "__main__":
    print("--- 1. Generating Dummy Data for Demonstration ---")

    # Generating synthetic data shapes matching target dataset scale
    num_samples = 1000
    n_mfcc, seq_len = 40, 174
    num_classes = 4  # e.g., Happy, Sad, Angry, Neutral

    # Synthetic Audio Features & Labels
    X_audio = np.random.randn(num_samples, seq_len, n_mfcc)

    # Synthetic Text Data
    sample_texts = [
        "I am so happy and excited about this result!",
        "This is terrible and makes me feel angry.",
        "I feel sad and lonely today.",
        "The project meeting is scheduled for tomorrow."
    ]
    raw_texts = np.random.choice(sample_texts, num_samples)

    # Labels
    labels = np.random.randint(0, num_classes, num_samples)

    print("\n--- 2. Processing Text Data ---")
    cleaned_texts = [preprocess_text(t) for t in raw_texts]

    # TF-IDF Vectorization
    vectorizer = TfidfVectorizer(max_features=500)
    X_text_vec = vectorizer.fit_transform(cleaned_texts).toarray()

    # Split Data
    y_cat = to_categorical(labels, num_classes=num_classes)

    (X_aud_tr, X_aud_te,
     X_txt_tr, X_txt_te,
     y_tr, y_te) = train_test_split(X_audio, X_text_vec, y_cat, test_size=0.2, random_state=42)

    print("\n--- 3. Training Audio Models (CNN & RNN) ---")
    cnn_model = build_cnn_emotion_model((seq_len, n_mfcc), num_classes)
    cnn_model.fit(X_aud_tr, y_tr, epochs=2, batch_size=32, verbose=0)
    print("CNN Model training complete.")

    rnn_model = build_rnn_emotion_model((seq_len, n_mfcc), num_classes)
    rnn_model.fit(X_aud_tr, y_tr, epochs=2, batch_size=32, verbose=0)
    print("RNN Model training complete.")

    print("\n--- 4. Training Text ML Models ---")
    y_tr_single = np.argmax(y_tr, axis=1)
    y_te_single = np.argmax(y_te, axis=1)
    text_results = train_text_classifiers(X_txt_tr, y_tr_single, X_txt_te, y_te_single)

    print("\n--- 5. Training & Evaluating Multimodal Fusion Model ---")
    multimodal_model = build_multimodal_model((seq_len, n_mfcc), X_text_vec.shape[1], num_classes)
    multimodal_model.fit(
        x={"Audio_Input": X_aud_tr, "Text_Input": X_txt_tr},
        y=y_tr,
        epochs=3,
        batch_size=32,
        verbose=0
    )

    # Evaluation
    predictions = multimodal_model.predict({"Audio_Input": X_aud_te, "Text_Input": X_txt_te})
    y_pred = np.argmax(predictions, axis=1)

    print("\n================ Multimodal Evaluation Metrics ================")
    print(classification_report(y_te_single, y_pred, target_names=[f"Class_{i}" for i in range(num_classes)]))

--- 1. Generating Dummy Data for Demonstration ---

--- 2. Processing Text Data ---

--- 3. Training Audio Models (CNN & RNN) ---
CNN Model training complete.
RNN Model training complete.

--- 4. Training Text ML Models ---
[Logistic Regression] Accuracy: 0.2200 | F1-Score: 0.2218
[Support Vector Machine] Accuracy: 0.2200 | F1-Score: 0.2218
[Random Forest] Accuracy: 0.2200 | F1-Score: 0.2218

--- 5. Training & Evaluating Multimodal Fusion Model ---
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

================ Multimodal Evaluation Metrics ================
              precision    recall  f1-score   support

     Class_0       0.00      0.00      0.00        45
     Class_1       0.17      0.02      0.04        47
     Class_2       0.50      0.10      0.17        49
     Class_3       0.30      0.95      0.46        59

    accuracy                           0.31       200
   macro avg       0.24      0.27      0.17       200
weighted avg       0.25      0.31      0.19       200



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
